# CreditWise Loan Approval System

## 1. Importing Libraries

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [ ]:
from src.preprocessing import preprocess_data, encoding,feature_sq
from src.eda import pie_chart, bar_graph
from src.train import lr,nb,knn,evaluate_model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

## 2. Loading Data

In [ ]:
cleaned_data = preprocess_data("../Data/loan_approval_data.csv")
cleaned_data.head(5)

## 4. Exploratory Data Analysis (EDA)

In [ ]:
pie_chart(cleaned_data,"Loan_Approved", ["Yes","No"],"Is Loan approved or not?")

In [ ]:
bar_graph(cleaned_data,"Education_Level")

In [ ]:
# analyze income
sns.histplot(
    data = cleaned_data,
    x = "Applicant_Income",
    bins=20
)

In [ ]:
sns.histplot(
    data = cleaned_data,
    x = "Coapplicant_Income",
    bins=20

)

In [ ]:
# outliers - box plots
fig, axes = plt.subplots(2, 2)

sns.boxplot(ax=axes[0, 0], data=cleaned_data, x="Loan_Approved",y="Applicant_Income")
sns.boxplot(ax=axes[0, 1], data=cleaned_data, x="Loan_Approved",y="Credit_Score")
sns.boxplot(ax=axes[1, 0], data=cleaned_data, x="Loan_Approved",y="DTI_Ratio")
sns.boxplot(ax=axes[1, 1], data=cleaned_data, x="Loan_Approved",y="Savings")

plt.tight_layout()

In [ ]:
# Credit Score with Loan Approved
sns.histplot(
    data=cleaned_data,
    x="Credit_Score",
    hue="Loan_Approved",
    bins=20,
    multiple="dodge"
)

In [ ]:
sns.histplot(
    data=cleaned_data,
    x="Applicant_Income",
    hue="Loan_Approved",
    bins=20,
    multiple="dodge"
)

## 5. Encoding

In [ ]:
data = encoding(cleaned_data)

In [ ]:
num_cols = data.select_dtypes(include="number")
corr_matrix = num_cols.corr()

plt.figure(figsize=(15, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)
num_cols.corr()["Loan_Approved"].sort_values(ascending=False)

## 6. Feature Engineering & Train-Test Split

In [ ]:
# Add or Tranform features
feature_sq(data,"DTI_Ratio_sq","DTI_Ratio")
feature_sq(data,"Credit_Score_sq","Credit_Score")


# df["Applicant_Income_log"] = np.log1p(df["Applicant_Income"])

X = data.drop(columns=["Loan_Approved", "Credit_Score", "DTI_Ratio","Applicant_ID"])
y = data["Loan_Approved"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

## 7. Model Building

In [ ]:
lr = lr()
lr.fit(X_train, y_train)
nb = nb()
nb.fit(X_train, y_train)
knn = knn()
knn.fit(X_train, y_train)

In [ ]:
y_pred_lr = lr.predict(X_test)
y_pred_nb =nb.predict(X_test)
y_pred_knn = knn.predict(X_test)

## 8. Evaluation

In [ ]:
evaluate_model("LR", y_test, y_pred_lr)
evaluate_model("NB", y_test, y_pred_nb)
evaluate_model("KNN", y_test, y_pred_knn)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, y_pred, title in zip(axes, 
                              [y_pred_lr, y_pred_nb, y_pred_knn], 
                              ["Logistic Regression", "Naive Bayes", "KNN"]):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm).plot(ax=ax)
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 9. Conclusion